Groundwater | Flow Modeling Track

# Step 2: Perceptual Model – The Limmat Valley Aquifer

Dr. Xiang-Zhao Kong & Dr. Beatrice Marti & Louise Noël du Payrat

In [ ]:
# Setup
import sys
import os

# Fix PROJ library path - must be set BEFORE importing geospatial libraries
# This prevents conflicts with other PROJ installations (e.g., anaconda)
import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import geopandas as gpd
import plotly.express as px

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file
import print_images as du
from map_utils import (
    display_groundwater_resources_map, 
    plot_model_area_map,
    display_concessions_map, 
    create_interactive_dem_map,
)
import climate_utils as cu
from climate_utils import get_and_plot_climate_data
import river_utils as ru
from river_utils import get_and_display_river_water_level_data, get_river_area
from perceptual_model_utils import get_and_plot_groundwater_levels

| **Core content** | ~45 minutes |
|:--|:--|
| **Optional: Detailed calculations & data exploration** | +30 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Identify** the key hydrogeological features of the Limmat Valley aquifer
2. **Explain** the main water balance components (recharge, river interaction, pumping, outflow)
3. **Estimate** rough magnitudes of groundwater fluxes using simple calculations
4. **Construct** a perceptual model diagram summarizing system understanding

## Prerequisites

Before starting this notebook, you should have:
- Completed [1_model_goal.ipynb](1_model_goal.ipynb) to understand the modeling objectives
- Basic understanding of Darcy's Law and aquifer properties
- Familiarity with water balance concepts

---

## Introduction

In this notebook, we build a **perceptual model** of the Limmat Valley aquifer - a mental map of how the groundwater system works before we translate it into a numerical model.

We follow the steps a professional groundwater modeler would take:
1. Understand the aquifer geometry and boundaries (Chapters 1-2)
2. Identify the main water fluxes: climate forcing (Chapter 3), rivers (Chapter 4), pumping (Chapter 5)
3. Review available monitoring data (Chapter 6)
4. Synthesize everything into a conceptual diagram (Summary)

> **💡 Tip: Use the Outline Panel**
> This is a long notebook with many sections. To keep track of where you are:
> - **In VS Code**: Open the Outline panel (View → Open View → Outline, or click the document icon in the sidebar)
> - **In JupyterHub/JupyterLab**: Open the Table of Contents panel (View → Table of Contents, or click the list icon in the left sidebar)
>
> The outline shows all section headings and lets you quickly jump between chapters.

We start with a generic perceptual model (see figure below) and refine it as we gather information.

In [ ]:
du.display_image(
    image_filename='perceptual_model_00_initial.png', 
    image_folder='1_perceptual_model',
    caption='Initial perceptual model of the Limmat valley aquifer (indicated in brown) which will be refined and used as a basis for a first long-term average water balance estimation. The purple arrows indicate important groundwater fluxes. Qin represent lateral inflows, while Qout represents outflows. Qpump represents groundwater extraction. Qrech and Qleak stand for areal recharge and leakage from surface water bodies respectively. ∆S stands for the storage change in the aquifer.'
)

We first aim to understand the hydrogeology of the Limmat Valley Aquifer. Next, we examine each water balance component to develop an initial estimate of the key fluxes within the system. Finally, we will revisit and refine the perceptual model as we prepare to construct a numerical model of the Limmat Valley Aquifer.

## 1 - The Limmat Valley Aquifer

The Limmat Valley Aquifer is a well-studied groundwater reservoir beneath the city of Zurich, Switzerland. According to Doppler and colleagues, it was formed during the last ice age as the Linth glacier retreated. The aquifer has no direct hydraulic connection to Lake Zurich in the east, where it is bounded by impermeable lake sediments and moraine material. It is further confined in the north and south by the side moraines of the Linth glacier. Lateral inflow of groundwater from the surrounding hills is expected. The aquifer is also hydraulically connected to the river Sihl in the east and the river Limmat in the north. Its hydraulic properties are highly heterogeneous due to a complex geological history shaped by various sedimentation and erosion events from the rivers Sihl and Limmat [\[1, 2\]](#References).

In the figure below, you see a screenshot of the [GIS-browser](https://geo.zh.ch/maps?x=2677655&y=1253620&scale=396217&basemap=arelkbackgroundzh) of the canton of Zurich [\[3\]](#References). The cyan-blue area in the center of the map shows the Limmat Valley Aquifer. The darker the color, the greater the thickness of the aquifer. The GIS-browser is only available in German, but you can use your browser's translation feature to view it in your preferred language.

In [ ]:
du.display_image(
    image_filename='GIS-browser_canton_Zurich.png', 
    image_folder='1_perceptual_model',
    caption='Printscreen of the GIS-browser of the canton of Zurich displaying the cantonal groundwater map. Source: https://www.gis.zh.ch/, accessed 2024-05-01'
)

> **Action point:**  
> Take a few minutes of your time to browse the available data layers in the GIS-browser. There are many! Focus on the layers that are relevant to the Limmat valley aquifer.

We now begin systematically reviewing the available information to refine our perceptual model of the Limmat Valley Aquifer. The GIS-browser will help us clarify the aquifer's geometry and inform how we represent it in the numerical model.

You can use the checklist below to track your progress.

## 2 - Aquifer Geometry

Most layers in the GIS-Browser of the Canton of Zurich are publicly available for download (see the "Datenbezug" button in the upper right corner of the map pane). We use this feature to import the relevant data into our JupyterHub environment.

### 2.1 - Bottom Topography

We have downloaded the aquifer thickness layer from the GIS-browser and clipped it to the area of interest (AOI) for the Limmat Valley Aquifer. Let's examine the bottom topography of the aquifer in the figure below.


In [ ]:
# This function downloads the groundwater map of the canton of Zurich and saves 
# it to the folder applied_groundwater_modelling_data in your home directory. 
gw_map_path = download_named_file(
    name='groundwater_map_norm', 
    data_type='gis', 
)


In [ ]:
# Create and display the interactive map
interactive_map = display_groundwater_resources_map(
    gw_map_path, zoom_level=11, 
    map_title="Groundwater resources map of the canton of Zurich",)
interactive_map

We can use contours of the aquifer thickness to build the bottom of the aquifer model (this is done in a future chapter : Notebook 4, model implementation).  

### 2.2 - Top Topography
The Limmat Valley Aquifer is unconfined. We can use the topography of the Limmat Valley to represent the top of the aquifer. [SwissTopo](../glossary.md#swiss-data-sources) provides digital elevation models ([DEMs](../glossary.md#data--technical-terms)) of Switzerland in various resolutions. We use the high-resolution swissALTI3D dataset with 2m resolution for our case study. This resolution is sufficient to capture important topographic features such as river beds, which is crucial for accurately modeling river-aquifer interactions. We downloaded the DEM from the [website of the Federal Office of Topography swisstopo](https://www.swisstopo.admin.ch/en/height-model-swissalti3d)[\[4\]](#References) and clipped it to the area of interest (AOI) of the Limmat Valley Aquifer (see figure below). If you are interested in learning more about the DEM and how to clip it to the AOI, you can read the
[processing_DEM.ipynb](../../_SUPPORT/src/scripts/scripts_limmat_data_preprocessing/processing_DEM.ipynb) notebook.

In [ ]:
dem_path = download_named_file(
    name='dem_hres',
    data_type='gis'
)

In [ ]:
m = create_interactive_dem_map(
    dem_path=dem_path, 
    zoom_start=12,
    container_title="DEM Controls",
    custom_title="High-resolution digital elevation model (swissALTI3D, 2m) in the area of interest. Background: OpenStreetMap"
)
m

The topography of the Limmat valley is quite flat, with a maximum elevation of about 400 m above sea level. With the 2m resolution of swissALTI3D, we can resolve important topographic features including river beds, which is essential for accurate river-aquifer interaction modeling.

River-aquifer interaction is predominantly driven by the stream bed elevation relative to the groundwater table. Therefore, an accurate representation of the streambed elevation in the numerical groundwater flow model is crucial. The high-resolution DEM allows us to better represent these features in our model. We will discuss the [river-aquifer interaction](#4---River-aquifer-interaction) in a later chapter.

### 2.3 - Lateral Model Boundaries
The groundwater map of the Limmat valley aquifer shows that the aquifer is not hydraulically connected to Lake Zurich in the east, but that it is bounded by impermeable lake sediments and moraine material. We will neglect aquifer parts east of the river Sihl as it is not the focus of the model goal, and we expect recharge from the river to dominate over flow from the east. The aquifer is further confined in the north and south by the side moraines of the Linth glacier. Lateral inflow of groundwater from the hills in the north and south is to be expected. The most common approach is to use a water balance approach on the hillsides to estimate the subsurface inflow from the hills. We'll address this in the chapter on [climate data](#3---Climate-Forcing). We do not consider the lateral inflow from the Sihl valley aquifer into the Limmat valley aquifer, as the inflow is rather small and parallel to the general flow direction in the Limmat valley.

Should you have to estimate the lateral inflow from the Sihl valley aquifer, you can apply Darcy's law under the Dupuit assumption. In practice, the organization tasking you to produce a groundwater flow model will typically provide you with estimates of [hydraulic conductivities (K)](../glossary.md#hydrogeological-parameters) from field measurements. As a first estimate, you can use the [hydraulic conductivities (K)](../glossary.md#hydrogeological-parameters) from the literature (e.g. Freeze and Cherry, 1979, shown in the figure below)[\[5\]](#References).

In [ ]:
du.display_image(
    image_filename='hydraulic_conductivities_by_Freeze.png', 
    image_folder='1_perceptual_model',
    caption='Hydraulic conductivity ranges as presented in the book <<Groundwater>> by Freeze and Cherry (1979). The ranges are based on the hydraulic conductivity of unconsolidated sediments. Hydraulic conductivity is typically known only in approximate orders of magnitude.'
)

<details>
<summary><strong>Optional: Theory reminder – The Dupuit Approximation</strong></summary>


> To apply Darcy's Law for calculating the total discharge through an unconfined aquifer, we often rely on a set of simplifying assumptions known as the *Dupuit approximation* (or Dupuit-Forchheimer assumption). These assumptions allow us to treat the complex three-dimensional flow problem as a simpler two-dimensional one by integrating flow over the entire aquifer thickness.
> 
> The key conditions that must be met for the Dupuit approximation to be valid are:
> 
> *   *The water table gradient is small:* The slope of the phreatic surface must be gente. This is the most critical assumption, as it implies that flow is almost entirely horizontal.
> *   *Flow is horizontal:* It is assumed that the hydraulic head is constant with depth for any given vertical line. This means equipotential lines are vertical, and therefore, the flow lines are horizontal.
> *   *The hydraulic gradient is equal to the slope of the water table:* The slope of the water table is assumed to be the hydraulic gradient (`i`) for throughout the saturated thickness of the aquifer.
> 
> When these conditions hold, we can use a modified form of Darcy's Law to calculate the discharge `Q` of the aquifer:
> 
> `Q = -K * h * B * (dh/dx)`
> 
> Where:
> *   `Q` is the discharge (e.g., m³/s)
> *   `K` is the hydraulic conductivity (e.g., m/s)
> *   `h` is the saturated thickness (m)
> *   `B` is the width of the aquifer perpendicular to the flow direction (m)
> *   `dh/dx` is the slope of the water table, which represents the hydraulic gradient.
> 
> In essence, the Dupuit approximation simplifies reality by ignoring vertical flow components, which is reasonable when the water table is nearly flat relatively to the saturated thickness.

</details>

For the outflow, a fixed-head boundary is typically chosen. This requires a known groundwater table at the boundary, e.g. a monitoring piezometer. Another option is to use the water level of a receiving stream or lake as a fixed-head boundary condition. The outflow boundary should be placed far enough away from the area of interest, so that the groundwater flow in the area of interest is not influenced by the boundary condition. For a regional model, this is typically several hundred meters to several kilometers away from the area of interest. 

In our case, we choose the groundwater level at piezometer 481 (indicated with a pink arrow in the figure below) as the fixed-head boundary condition. This relieves us from the need to implement the more complex aquifer geometry and river-aquifer interaction further downstream in the Limmat valley aquifer. We can use the isoline of equal groundwater level at 390 m a.s.l. from the GIS browser to define the outflow boundary.

In [ ]:
du.display_image(
    image_filename='Detail_outflow.png', 
    image_folder='1_perceptual_model',
    caption='Detail of the outflow boundary condition at piezometer 481 (purple arrow). The groundwater level at this piezometer is used as a fixed-head boundary condition for the groundwater flow model. The outflow boundary is placed far enough away from the area of interest, so that the groundwater flow in the area of interest is not influenced by the boundary condition.'
)

Note that the isolines of equal groundwater levels are not perfectly perpendicular to the flow direction but slightly curved at the aquifer boundaries (particularly visible at the northern boundary). This indicates lateral groundwater inflow from the north.  

We can now make a rough estimate of the groundwater flux at the western outflow boundary of the Limmat Valley Aquifer using the Dupuit's assumption. We use the groundwater levels isolines of 389 m a.s.l. and 391 m a.s.l. to estimate the hydraulic gradient. The distance between the two isolines can be measured in a GIS tool and is about 780 m. The thickness of the aquifer in that location is between 2 m and 10 m, and the width is approximately 1 km. We estimate the hydraulic conductivity to be around 10^-2 m/s so we can estimate the hydraulic gradient as follows:


In [ ]:
outflow_lower_bound = (391 - 389) / 780 * 2 * 1000 * 0.01 * 3600 * 24 # m3/day
outflow_upper_bound = (391 - 389) / 780 * 10 * 1000 * 0.01 * 3600 * 24 # m3/day

print(f"Estimated groundwater outflow at the western boundary of the Limmat valley aquifer: {outflow_lower_bound:.0f} m3/day to {outflow_upper_bound:.0f} m3/day")

We now have an understanding of the geometry of the aquifer. Let's simplify update our virtual perceptual model with the information we have gathered so far: 

- The aquifer does not have a direct hydraulic connection to lake Zurich in the east. 
- It is probably heavily influenced by the river Sihl in the east and the river Limmat in the north.
- Since groundwater flow in river valleys generally follows the topography, we can assume that the groundwater flow in the Limmat valley aquifer is directed from south-east to west. This general flow direction is corroborated by the groundwater flow arrows which become visible when we zoom in on the GIS-browser map (visit the GIS browser to zoom in).
- The aquifer thickness is highly variable. To adequately represent the known geometry of the aquifer, a 3D model with a high spatial resolution is required. We will first opt for a 2D model with a simplified geometry for the case study work to keep the model lean and fast on JupyterHub.
- The simplified geometry will show a larger thickness in the south-east with a gradual thinning towards the west. The thickness will be represented by a single layer with a variable thickness. We will use the thickness values from the GIS-browser to define the thickness of the aquifer in our model whereby we will have to map the isolines of groundwater thickness to the model grid.
- The aquifer extends beyond the boundaries of the city of Zurich. Since we are interested mainly in applying our numerical experiment in the city region, this means that we will have to make assumptions about the outflow boundary of the aquifer.

 

### 2.4 - Drawing the model boundary polygon
We can now draw the model boundary polygon manually in QGIS and export it as a geopackage layer. Given the complex aquifer thickness, it is the fastest method. This layer will define the model area in our numerical model.

If you need instructions of how to use QGIS to draw and edit a polygon, please refer to resources available online. 

In [ ]:
model_boundary_path = download_named_file(
    name='model_boundary',
    data_type='gis'
)

# Get the area of the model domain
gdf_boundary = gpd.read_file(model_boundary_path)
model_area_m2 = gdf_boundary.geometry.area.sum()
model_area_km2 = model_area_m2 / 1e6  # Convert to km²
print(f"Model area: {model_area_km2:.2f} km²")  

plot_model_area_map(
    gw_depth_path=gw_map_path, 
    model_boundary_path=model_boundary_path,
    custom_title="Model region with the model boundary in black. Base layer by OpenStreetMap.",
    basemap="osm", 
    basemap_alpha=0.9
)

## 3 - Climate Forcing

### 3.1 - Areal Recharge
We get climate data from the Swiss Federal Office of Meteorology and Climatology (MeteoSwiss). The data is available for viewing on [MeteoSwiss](https://www.meteoswiss.admin.ch/services-and-publications/applications/measurement-values-and-measuring-networks.html#param=messnetz-klima&table=false&station=SMA&compare=y&chart=year) website [\[6\]](#References) and was downloaded via the [opendata.swiss plattform](https://opendata.swiss/en/dataset/klimanormwerte) [\[7\]](#References). It is available in the data directory of the case study repository. 
The closest station to the Limmat Valley Aquifer is Fluntern. We use data from there, which is available for the time period 1991-2020. The figure below shows the monthly average temperature and precipitation for the Fluntern station. 

In [ ]:
# Download, read, and plot climate data for the Fluntern station
climate_data_path, climate_norms = get_and_plot_climate_data(
    custom_title="Climate data for the Fluntern station."
)

> **🤔 Think about it**  
> We are looking at the climate data mostly to get an idea about groundwater recharge. In an aquifer located in a densely populated area, we have to consider that the recharge is not only influenced by the climate but also by human activities. What do you think are the most important human activities that influence groundwater recharge in this area?
>
> Hint: Consider how urbanization, land use changes, and water management practices can affect infiltration and runoff patterns.

Now we need to get from climate data to net recharge at the groundwater table, taking into account the mostly built-up land cover. You have several options to do this which are listed below by increasing complexity :  

- Use net recharge literature values in the project area, or reported fractions from precipitation in comparable areas ( ie. of similar climatic and land cover conditions)
- Estimate net recharge using a water balance approach (precipitation minus evaporation)
- Use a soil water balance model (e.g. Hydrus) or a land surface model  
- Simulate the water flow through the unsaturated zone in Modflow (requires high-resolution 3D grid for numerical stability)

We start out with the less complex option. Lehner states that between 5% and 15% of the precipitation infiltrates into the aquifer [\[8\]](@references). We will use a value of 10% as a first approximation, ie. 110 mm/year.
The rest of the precipitation evaporates (approximately 20% according to Qui et al. (2023)) or runs off into the Sihl or the Limmat (via the rainwater drainage system). 

Once the area of the model domain is known, we can calculate the total volume of areal recharge to the aquifer. 

### 3.2 - Lateral Inflow from Hills

The lateral inflow from the hills is estimated using a water balance approach on the hill sides that contribute to the aquifer. 

Information needed are :
- climate data (as for the areal recharge in the previous paragraph)
- hills area, which is estimated using QGIS (see the figure below)

We assume that 10% of the precipitation in a hill area infiltrates and is transported, eventually becoming lateral inflow into the aquifer.

In [ ]:
du.display_image(
    image_filename='rough_first_manual_catchment_delineation_to_estimate_area_for_lateral_inflow.png', 
    image_folder='1_perceptual_model',
    caption='Manual catchment delineation to estimate area for lateral inflow. The area is delineated using the height model of the area (here DHM200 by Swisstopo). The area is used to estimate the lateral inflow from the hills in the north and south of the Limmat valley aquifer. The delineation is not precise and should be refined in a later step. The area relevant for the lateral inflow estimation is 15 km2 in the south and 11 km2 in the north of the Limmat valley aquifer.'
)

Please note, this manual delineation is sufficient for a first rough estimation. A proper catchment delineation would require a more detailed analysis of the topography and the hydrological conditions in the area. You find step-by-step instructions on how to delineate a catchment in many places on the internet, for example here: [https://hydrosolutions.github.io/caham_book/geospatial_data.html#sec-catchment-delineation](https://hydrosolutions.github.io/caham_book/geospatial_data.html#sec-catchment-delineation). 



In [ ]:
inflow_south = 0.1 * 15e6  # m3/year
inflow_north = 0.1 * 11e6  # m3/year

print(f"Estimated lateral inflow from the south: {(inflow_south/1000000):.1f} 10^6 m³/year")
print(f"Estimated lateral inflow from the north: {(inflow_north/1000000):.1f} 10^6 m³/year")

> **Think about it:**. 
> What type of boundary condition would you use to represent the lateral inflow from the hills in a steady state numerical model? Why?  
> 
> Hint: Remember, we know 3 types of boundary conditions: no-flow, fixed-head, and specified-flux.

## 4 - River-aquifer interaction
In most unconsolidated rock aquifers in river valleys, river-aquifer interaction is a key process that influences groundwater levels and flow patterns. The interaction between the river and the aquifer can lead to changes in water levels in the aquifer, which can affect the availability of groundwater resources. We therefore look at this process in more detail:

1. Understand the river geometry and its interaction with the aquifer (chapter [4.1](#41---introduction-to-river-geometry--recharge)).
2. Analyze monitoring data for river discharge and river water levels (chapter [4.2](#42---understanding-monitoring-data-river-discharge--river-water-levels)).
3. Estimate the leakage flux between the river and the aquifer (chapter [4.3](#43---estimating-the-leakage-flux)).

### 4.1 - Introduction to River Geometry & Recharge
The river-aquifer interaction is an important component of the groundwater flow model. The geometry of the river and its interaction with the aquifer influence both the direction of groundwater flow and the groundwater levels in the aquifer.

In practice, a modeler uses measured profiles of the riverbed to represent the river geometry in the model. For this course, we do not have access to riverbed profiles so we must make some assumptions about the river geometry.

> **💡 Tip: Visit your model area**
> If you are based in Zurich, you can visit the rivers. Visiting your model area is a great way to get a feel for the site and understand the hydrological processes at play. 

#### Optional 4.1.1 Interactive Exploration of River-Aquifer Interaction

The figure below provides an interactive tool to help you visualize the complex relationship between a river and an adjacent unconfined aquifer. It consists of two main parts:

1.  **Left Plot (Physical Cross-Section):** This shows a physical model of the river and the groundwater system. You can see the water level in the river (`H_riv`) and the water table in the aquifer (`H_aq`).
2.  **Right Plot (Flux Relationship):** This conceptual graph shows the calculated flux (flow rate, Q_riv) between the river and the aquifer as a function of the aquifer head. The red dot indicates the current state of the system shown on the left.

**How to Use This Figure:**

Use the two sliders below the plot to change the **Aquifer Head (`H_aq`)** and the **River Stage (`H_riv`)**. As you move the sliders, observe how the system changes in both plots. Pay close attention to the title, the equation, the direction of the flow (indicated by the arrow), and the position of the red dot on the flux curve.

**Scenarios to Explore:**

Try to create the following three fundamental conditions:

1.  **Gaining Stream:**
    *   **How to create it:** Set the aquifer head (`H_aq`) to be *higher* than the river stage (`H_riv`).
    *   **What to observe:** Water flows from the aquifer into the river. The flux (`Q_riv`) is negative (flow *out of* the groundwater). The head difference (`ΔH`) is between `H_aq` and `H_riv`. Nb : on the right plot, the red dot is in the negative flux region.

2.  **Losing Stream (Connected):**
    *   **How to create it:** Set the river stage (`H_riv`) to be *higher* than the aquifer head (`H_aq`), but keep `H_aq` *above* the riverbed bottom (`R_bot`, the dashed brown line).
    *   **What to observe:** Water flows from the river into the aquifer. The flux is positive and its magnitude depends directly on the head difference (`ΔH`). As you lower `H_aq`, the flux increases. The red dot on the right plot moves down the sloped part of the curve.

3.  **Losing Stream (Disconnected):**
    *   **How to create it:** Lower the aquifer head (`H_aq`) until it is *below* the riverbed bottom (`R_bot`).
    *   **What to observe:** A yellow "Vadose Zone" appears between the riverbed and the water table, indicating they are no longer directly connected. The flow from the river now seeps downward at a constant, maximum rate. The flux no longer depends on `H_aq`. Notice that the head difference (`ΔH`) for the calculation is now between `H_riv` and `R_bot`. On the right plot, the red dot moves onto the vertical part of the curve, showing that the flux remains constant even if you continue to lower `H_aq`.

In [ ]:
ru.plot_river_aquifer_interaction(
    custom_title="Illustration of river-aquifer interaction and the corresponding flux vs. head relationship."
)

### 4.2 - Understanding Monitoring Data: River Discharge & River Water Levels

#### Optional 4.2.1 - Where to get the data from
The BAFU offers access to many hydrological data for Switzerland. You can find surface water monitoring sites under [map.geo.admin.ch](https://map.geo.admin.ch), under layer *Hydrological gauging stations* [\[10\]](#References). When you zoom in to the Limmat Valley Aquifer (see figure below), you find the gauging stations of the rivers Sihl and Limmat. The Limmat gauging station located in Baden is downstream of a run-by-the-river hydropower plant, and therefore not relevant for our study. Relevant stations are the Sihl gauging station in the city of Zurich (right upstream of the confluence with the river Limmat) and the Limmat gauging station also in the city of Zurich (right downstream of the confluence with the Sihl). 

In [ ]:
du.display_image(
    image_filename='BAFU_HydrologicalGaugingStations_closeup.png', 
    image_folder='1_perceptual_model',
    caption='Locations of hydrological gauging stations near the Limmat valley aquifer.'
)

A click on the gauaging station brings you directly to the station site made available by [BAFU](../glossary.md#swiss-data-sources). The gauging station on the Sihl is called [*Sihl - Zürich, Sihlhölzli*](https://www.hydrodaten.admin.ch/de/seen-und-fluesse/stationen-und-daten/2176) and has ID 2176 and the gauging station on the river Limmat is called [*Limmat - Zürich, Unterhard*](https://www.hydrodaten.admin.ch/de/seen-und-fluesse/stationen-und-daten/2099) and has ID 2099. The IDs are unique numeric station identifiers, and are typically required to retrieve data from data repositories or APIs. 

> **🤔 Think about it**  
> For surface water balancing, discharge is the important variable. However, when it comes to flooding or river-aquifer interaction, the water level is the more important variable. Why do you think that is?
>
> Hint: Consider how water levels relate to hydraulic gradients and flow directions between rivers and aquifers.

From the station sites, we see current water levels, typically over the past 7 days but no water level dynamics. We had to request this data from BAFU. Please note that data for recent years data is provisional and subject to change.

Additional river gauging stations are maintained by the cantons. In our case however, the cantonal office [AWEL](../glossary.md#swiss-data-sources) (cantonal office for waste, water, energy and air) [\[11\]](#References) does not maintain additional runoff gauges in the are of interest. 

#### 4.2.2 - Summary of Water Level Data
Let's have a look at the data (Figures and Table below). 


In [ ]:
# Download, plot, and summarize river water level data
river_data_path, summary = get_and_display_river_water_level_data(
    start_year=2010,
    end_year=2020,
    figure_number=13
)

### 4.3 - Estimating the Leakage Flux
We need to compare river water levels and groundwater levels, to determine if the streams are gaining or losing water relatively to the aquifer and to estimate the water flux between the rivers and the aquifer. We use groundwater levels extracted form the mean and max isohypse layers in the GIS-browser at the locations of the river gauging stations. Based on this information, we can draw a conceptual figure of the river cross-sections with the river-aquifer interaction at the two gauging stations (see figures below).

In [ ]:
# Sihl River Data
sihl_gw_mean = 400.8
sihl_gw_high = 403.5
sihl_river_mean = 412.347
sihl_river_high = 413.439
sihl_depth_mean = 0.3

# Limmat River Data
limmat_gw_mean = 399.5
limmat_gw_high = 400.5
limmat_river_mean = 400.285
limmat_river_high = 401.822
limmat_depth_mean = 0.7

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14))
fig.suptitle("River and Groundwater Level Cross-Sections", fontsize=16, y=0.95)

# Plot Sihl
ru.plot_cross_section(ax1, "a) River Sihl", sihl_gw_mean, sihl_gw_high, sihl_river_mean, sihl_river_high, sihl_depth_mean)

# Plot Limmat
ru.plot_cross_section(ax2, "b) River Limmat", limmat_gw_mean, limmat_gw_high, limmat_river_mean, limmat_river_high, limmat_depth_mean)

plt.tight_layout(rect=[0, 0, 1, 0.93]) # Adjust layout to make space for suptitle
plt.show()

From the two conceptualized cross-sections at the gauge location on the river Sihl we see that we are dealing with a losing stream which is disconnected from the groundwater table. We therefore have a constant flux from the river into the aquifer. The flux is not influenced by the groundwater table in the aquifer. The flux is influenced only by the river stage and the river bed elevation.
The river Limmat is also a loosing stream at the location of the gauge. However, from the conceptualized cross-section we see that the river Limmat is very likely connected to the groundwater table. We have to account for a capillary fringe of up to 30 cm (depending on the pore size distribution of the soil matrix). The flux from the river into the aquifer is influenced by the groundwater table in the aquifer. The flux is further influenced by the river stage and the river bed elevation.

The river-aquifer interaction is limited by the hydraulic conductivity and the thickness of the clogging layer (colmation layer), a bio-geochemically highly active sediment layer which forms on river beds over time. The clogging layer is typically a few centimeters thick and has a significantly lower hydraulic conductivity than the underlying aquifer material. It therefore limits the flux from the river into the aquifer. Doppler and colleagues [\[12\]](#References) discuss the implementation of the river-aquifer interaction in their groundwater flow model of the Limmat valley aquifer. 

$$q_{riv} = \frac{K_{Schmutzschicht}}{d_{Schmutzschicht}} \cdot (H_{riv} - H_{aq}) = l_{leakage} \cdot (H_{riv} - H_{aq})$$

Where:
- $q_{riv}$ is the flux from the river into the aquifer ($m$ $s^{-1}$)
- $K_{Schmutzschicht}$ is the hydraulic conductivity of the Schmutzschicht ($m$ $s^{-1}$)
- $d_{Schmutzschicht}$ is the thickness of the clogging layer of the river bed ($m$) 
- $l_{leakage}$ is the leakage coefficient ($s^{-1}$)
- $H_{riv}$ is the river water level ($m$)
- $H_{aq}$ is the groundwater level in the aquifer ($m$)

Doppler and colleages calibrated leakage coefficients of $1.3 \cdot 10^{-6}$ $1/s$ for the river Sihl and between $3.5 \cdot 10^{-7}$ $1/s$ and $3.5 \cdot 10^{-5}$ $1/s$ for the river Limmat.  

To simplify the first rough estimation of groundwater recharge from river infiltration, we assume :
- disconnected streams for both rivers 
- an average river water level of 0.3 m to 0.5 m , respectively for the rivers Sihl and Limmat

This yields the following specific fluxes from the rivers to the aquifer: 



In [ ]:
q_riv_sihl = 1.3e-6 * (sihl_river_mean - sihl_gw_mean)  # m/s
q_riv_limmat = 3.5e-6 * (limmat_river_mean - limmat_gw_mean)  # m/s

print(f"Estimated specific flux from Sihl to aquifer: {q_riv_sihl:.2e} m/s")
print(f"Estimated specific flux from Limmat to aquifer: {q_riv_limmat:.2e} m/s")

The flux from the Limmat is expected to be to the upper end of the range, because the river is in some stretches connected to the groundwater table. The aquifer might exfiltrate into the Limmat in downstream stretches but those are far away from our focus area.     

The area of the rivers Sihl and Limmat we can estimate from the geopackage layer "Oeffentliche Oberflaechengewaesser" (see the figure below) provided by the Department of Geoinformation of the Canton of Zurich ([https://opendata.swiss/en/dataset/offentliche-oberflachengewasser](https://opendata.swiss/en/dataset/offentliche-oberflachengewasser), accessed 2025-07-06) using QGIS. Once we have determined the model boundary and estimated the area of the river beds within the model boundary, we can estimate a recharge volume from the rivers to the aquifer. 

In [ ]:
# Get river areas within model boundary
river_areas = get_river_area(gw_map_path=gw_map_path, show_figures=False)
sihl_area = river_areas['sihl_area']
limmat_area = river_areas['limmat_area']

In [ ]:
river_sihl_inflow = sihl_area * q_riv_sihl * 365 * 24 * 3600  # m³/year
river_limmat_inflow = limmat_area * q_riv_limmat * 365 * 24 * 3600  # m³/year

print(f"Estimated groundwater recharge from Sihl: {(river_sihl_inflow/1000000):.1f} 10^6 m³/year")
print(f"Estimated groundwater recharge from Limmat: {(river_limmat_inflow/1000000):.1f} 10^6 m³/year")

## 5 - Groundwater pumping

The Limmat Valley aquifer is heavily used for groundwater abstraction—for drinking water, industrial cooling, and heat pumps. Since actual pumping rates are not publicly available, we estimate abstraction from **concession data** (maximum permitted rates) in the cantonal GIS-browser.

**Key findings** (details in optional sections below):
- **Hardhof well field** dominates extraction: ~7 million m³/year for Zurich's drinking water supply
- **Thermal use** (heat pumps) is widespread but **non-consumptive**—water is extracted and reinjected
- Actual pumping can be lower than concessioned amounts (e.g., Hardhof uses only ~13% of its permitted volume)

### 5.1 - Active concessions in the model area

In [ ]:
# Download and load wells data (Grundwasserfassungen)
wells_path = download_named_file(name='wells', data_type='gis')
wells_gdf = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')

# Clip to model boundary and add concession ID
wells_gdf = wells_gdf.clip(gdf_boundary)
wells_gdf['concession_id'] = wells_gdf['GWR_ID'].str.split('_').str[0]

# Filter for active wells only (remove decommissioned and unused)
is_active = ~(
    wells_gdf['BESCHREIBUNG'].str.contains('aufgehoben', case=False, na=False) |
    wells_gdf['BESCHREIBUNG'].str.contains('ungenutzt', case=False, na=False)
)
wells_active = wells_gdf[is_active].copy()

# Summary
print(f"Active concessions in model area: {wells_active['concession_id'].nunique()}")
print(f"\nBreakdown by size category:")
print(wells_active.groupby('BESCHREIBUNG')['concession_id'].nunique().to_string())

The dataset uses German abbreviations for well usage types: **TW** = drinking water, **BW** = process water, **KW** = cooling water, **WPG** = groundwater heat pump. Wells sharing the same concession ID belong to the same water right.

Most concessions are for **thermal use** (heat pumps), which is non-consumptive: water is extracted, used for heating/cooling, and reinjected nearby. For our water balance, we focus on **consumptive uses** (drinking water, industrial water) in the large concession category (>3000 L/min).

We observe intensive thermal use of groundwater in the Limmat Valley aquifer. Thermal use is typically **non-consumptive**: water is abstracted in one well and returned to the aquifer in another after use.

For our water balance, we focus on the **Hardhof well field** (concession b1-71), which dominates groundwater extraction in the model area.

**Hardhof well field** is one of four pillars of the drinking water supply of the City of Zurich [\[13\]](#references) and contributes an average of 15% of the annual supply (most comes from treated lake water). The well field operates as a **managed aquifer recharge (MAR) system**: riverbank filtrate is extracted along the River Limmat, **artificially infiltrated** into 3 recharge basins and 12 infiltration wells, and finally collected by horizontal wells for distribution. According to the statistical yearbook of the City of Zurich, the water works produce around **7 million m³ of groundwater per year** [\[14\]](#references).

**Important for modeling:** Most of the 7 million m³/year extracted is riverbank filtrate that has been artificially recharged into the aquifer. If we implement only the pumping without the artificial recharge, the model will not converge. For our simplified model, we therefore work with **net abstraction** (pumping minus artificial recharge), which is close to zero or slightly negative (net inflow to the aquifer).

The concession allows 104,000 L/min, but only about 13% is actually extracted—illustrating that concession data represents maximum permitted volumes, not actual use.

> **Suggestion:** If you are in Zurich, we encourage you to visit the Hardhof well field on one of their free public guided tours.

### Optional: 5.2 - Detailed analysis of medium concessions

The following cells analyze medium-sized concessions (300-3000 L/min) to ensure we don't miss significant fluxes.

In [ ]:
id_col    = "concession_id"   # adjust if different
desc_col  = "BESCHREIBUNG"
use_col   = "NUTZART"

phrase_medium      = "Grundwasserfassung mit Ertrag 300 - 3000 l/min"
phrase_recharge  = "Grundwasseranreicherungsanlage, Rückversickerung, Sickergalerie"

# 1. Keep only wells with a non-empty NUTZART (still “active” set assumed already)
wells_valid = wells_active[
    wells_active[use_col].notna() &
    wells_active[use_col].astype(str).str.strip().ne("")
].copy()

# 2. Identify medium-yield wells
mask_medium = wells_valid[desc_col].str.contains(phrase_medium, case=False, na=False)
medium_yield_wells = wells_valid[mask_medium]

# 3. Concessions having at least one medium-yield well
concessions_with_medium = (
    medium_yield_wells[id_col]
    .dropna()
    .astype(str)
    .unique()
)

# 4. Subset ALL wells belonging to those concessions (this is the expanded set)
medium_concessions = wells_valid[
    wells_valid[id_col].astype(str).isin(concessions_with_medium)
].copy()

# 5. Classify well role inside those concessions
medium_concessions["WELL_GROUP"] = np.select(
    [
        medium_concessions[desc_col].str.contains(phrase_medium, case=False, na=False),
        medium_concessions[desc_col].str.contains(phrase_recharge, case=False, na=False),
    ],
    ["MediumYield", "Recharge"],
    default="OtherWithinConcession"
)

# (Optional) If you want to keep only wells that are either HighYield or Recharge
# plus all other wells in those *same* concessions (already done above).
# If you instead want to drop the 'OtherWithinConcession' group, uncomment:
# medium_concessions = medium_concessions[medium_concessions['WELL_GROUP'] != 'OtherWithinConcession'].copy()

print(f"Concessions with medium-yield wells: {len(concessions_with_medium)}")
print("Counts by WELL_GROUP:")
print(medium_concessions["WELL_GROUP"].value_counts())

# Count the number of concessions for TW and BW
print("Number of concessions for TW and BW:")
print(medium_concessions[medium_concessions[use_col].isin(["TW", "BW", "TW, BW"])][id_col].nunique())

display_concessions_map(
    medium_concessions, 
    boundary_gdf=gdf_boundary, 
    map_title="Wells belonging to medium concessions (300 - 3000 l/min) in the model area colored by concession ID and marked by use type. Small and large active concessions are not shown in this map."
)


In [ ]:
# List the active wells in medium concessions which are used for drinking water and industrial water use
active_wells_medium_concessions = medium_concessions[medium_concessions[use_col].isin(["TW", "BW", "TW, BW"])]
print(active_wells_medium_concessions[['GWR_ID', 'FASSBEZ', 'FASSART', 'NUTZART']])

By clicking on a well in the GIS-Browser, we can access slightly more information than what is provided in the geodata layer available here. The Lochergut concession (b010063) is for 520 L/min, and the Schlachthof concession is for 370 L/min. The concessioned pumping rate for concession b010113 is 2000 L/min. The two drinking water wells north of the Limmat (concessions n010039_01 and n010085_01) allow pumping of 1200 L/min and 3000 L/min, respectively.

Since the two drinking water wells north of the river are most likely pumping riverbank filtrate and are located far downstream of our focus area, we will not implement them for now. As we observed at Hardhof, only a fraction of the concessioned amount is typically extracted. For the purpose of this model, we will assume 40%. Under this assumption, the medium-sized concessions south of the Limmat would amount to about 600,000 m³/year (less than 10% of the Hardhof abstractions).

In [ ]:
# Hardhof well field: Managed Aquifer Recharge (MAR) system
# The well field extracts ~7 million m³/year but most of this is riverbank filtrate
# that has been artificially recharged into the aquifer first.

outflow_pumping = 7.0e6  # m³/year - gross extraction from Hardhof
inflow_artificial_recharge = 1.1 * outflow_pumping  # m³/year - artificial recharge slightly exceeds pumping

# Net effect on aquifer (what we implement in the simplified model)
net_hardhof = outflow_pumping - inflow_artificial_recharge  # m³/year (negative = net inflow)

print(f"Hardhof gross pumping:        {outflow_pumping/1e6:.1f} 10^6 m³/year")
print(f"Hardhof artificial recharge:  {inflow_artificial_recharge/1e6:.1f} 10^6 m³/year")
print(f"Hardhof net effect:           {net_hardhof/1e6:.1f} 10^6 m³/year (negative = net inflow to aquifer)")
print(f"\nFor the simplified model, we use net_hardhof = {net_hardhof/1e6:.1f} 10^6 m³/year")

## 6 - Monitoring the Limmat Valley Aquifer

To gain a first impression of the groundwater levels in the Limmat valley aquifer, we will have a look at the groundwater map.

Several authorities do groundwater monitoring in the Limmat valley aquifer. We will start with the federal office for the environment (FOEN) which maintains a network of groundwater observation wells. You find an overview over the available groundwater monitoring sites at [https://map.geo.admin.ch/](https://map.geo.admin.ch/) in layer *Groundwater level/spring discharge* [\[13\]](#References). One monitoring well maintained by FOEN is located in the Limmat valley aquifer but far downstream of our area of interest in the city center. Further, this well is a drinking water production well and can therefore not be used as an outflow boundary.  

Few groundwater observation wells are operated by the cantonal office of the environment ([AWEL](../glossary.md#swiss-data-sources)) [\[14\]](#References) (see figure below).

In [ ]:
du.display_image(
    image_filename='GWmonitoring_locations_AWEL.png', 
    image_folder='1_perceptual_model',
    caption='Monitoring wells in the Limmat valley aquifer. Yearbook sheets for each site can be accessed through the popup window from each site. Source: https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/grundwasserstaende.html, accessed 2025-05-01.'
)

### 6.1 - Groundwater Level Trends

To verify that the aquifer is in steady state, we analyze groundwater level data from AWEL cantonal monitoring wells. If the system is in steady state, we expect no significant long-term trends.

In [ ]:
# Load and plot groundwater level trends from AWEL monitoring wells
gw_ts_path, gw_annual = get_and_plot_groundwater_levels()

### 6.2 - Steady-State Assumption

Looking at the annual mean groundwater levels over the past decades, no significant long-term trends are visible. We can assume that the groundwater levels have remained relatively stable over this period and that the system is in **steady state**.

$$ \Delta S = 0 $$

This justifies using a steady-state model for our initial groundwater flow simulation.

## Summary

In [ ]:
# Display figure with updated fluxes
du.display_image(
    image_filename='perceptual_model_01_final.png', 
    image_folder='1_perceptual_model', 
    caption='Final perceptual model of the Limmat valley aquifer with estimated fluxes. The purple arrows indicate important groundwater fluxes. Qin represent lateral inflows, while Qout represents outflows. Qpump represents groundwater extraction. Qrech and Qleak stand for areal recharge and leakage from surface water bodies respectively. ∆S stands for the storage change in the aquifer.'
)

In [ ]:
# Calculate the missing fluxes for the water balance
net_recharge = 0.2 * model_area_km2 * 1000000  # m³/year, assuming 200 mm/year net recharge
river_sihl_inflow = sihl_area * q_riv_sihl * 365 * 24 * 3600  # m³/year
river_limmat_inflow = limmat_area * q_riv_limmat * 365 * 24 * 3600  # m³/year
gw_outflow_west = (outflow_lower_bound + outflow_upper_bound) / 2 * 365  # m³/year, average of lower and upper bound

# Create a summary table with the most relevant in- and outflows of the model domain
# Note: Hardhof MAR system has net_hardhof < 0, meaning net inflow to aquifer
# We add the absolute value to inflows since it represents a net gain
inflow_summary = {
    "Component": [
        "Net areal recharge to the aquifer (R)",
        "Lateral inflow from the south",
        "Lateral inflow from the north",
        "River Sihl (net inflow)",
        "River Limmat (net inflow)",
        "Hardhof MAR system (net inflow)*",
    ], 
    "Value (10^6 m³/year)": [
        round(net_recharge / 1e6, 1),
        round(inflow_south / 1e6, 1),
        round(inflow_north / 1e6, 1),
        round(river_sihl_inflow / 1e6, 1),
        round(river_limmat_inflow / 1e6, 1),
        round(abs(net_hardhof) / 1e6, 1),  # net_hardhof is negative, so abs() gives inflow
    ]
}
outflow_summary = {
    "Component": [
        "Groundwater outflow in the west",
    ], 
    "Value (10^6 m³/year)": [
        round(gw_outflow_west / 1e6, 1),
    ]
}

inflow_summary_df = pd.DataFrame(inflow_summary)
outflow_summary_df = pd.DataFrame(outflow_summary)

print("Inflows:")
print(inflow_summary_df)
print("")
print("Outflows:")
print(outflow_summary_df)
print("")
print(f"Sum of inflows: {inflow_summary_df['Value (10^6 m³/year)'].sum():.1f} 10^6 m³/year")
print(f"Sum of outflows: {outflow_summary_df['Value (10^6 m³/year)'].sum():.1f} 10^6 m³/year")
print(f"Difference (inflows - outflows): {(inflow_summary_df['Value (10^6 m³/year)'].sum() - outflow_summary_df['Value (10^6 m³/year)'].sum()):.1f} 10^6 m³/year")
print("")
print("*Hardhof extracts ~7 10^6 m³/year but artificially recharges ~7.7 10^6 m³/year,")
print(" resulting in a net inflow to the aquifer. For the simplified model, we use this net value.")

We currently observe much higher inflows than outflows. This is acceptable for this first estimate—the purpose of this exercise was to gain a rough understanding of the groundwater budget in the Limmat Valley aquifer.

Note that the Hardhof well field appears as a **net inflow** because it operates as a Managed Aquifer Recharge (MAR) system: the artificial recharge slightly exceeds the pumping. For the simplified model, we implement only the net effect rather than modeling both pumping and recharge separately.

We already know that river–aquifer interactions carry the highest uncertainty and cannot be adequately estimated with the data currently available. In a real-world project, you would need to revisit your river–aquifer interaction model and evaluate whether the chosen parameterization can be adjusted within physically meaningful limits, or whether additional data is required.

### Uncertainty of Flux Estimates

Our perceptual model provides a first approximation of groundwater fluxes in the Limmat Valley aquifer. However, all estimated fluxes are subject to considerable uncertainty that should be explicitly acknowledged. Below we summarize the main sources of uncertainty (note that the uncertainty ranges are themselves approximate):

**Net areal recharge (±50%)** : Applying a uniform value of 10% of precipitation across the entire model domain is a major simplification. Recharge in urban areas is highly variable due to impervious surfaces, leaking infrastructure, and artificial drainage. Reported values for urban recharge in the literature range from 5% to 30% of precipitation.

**Lateral inflow from hills (±40%)** : Our simplified catchment delineation and assumed recharge coefficient introduce uncertainty. Actual subsurface flow paths and recharge rates in hillslopes likely vary both spatially and temporally.

**River–aquifer interaction (±70%)** : This component carries the greatest uncertainty due to:
- Simplified representation of river geometry and bed elevation
- Assumptions about leakage coefficients derived from literature
- Spatial variability of riverbed conductance (e.g., clogging layer thickness)
- Temporal dynamics of river stages not represented in the steady-state model
- Uncertainty about connectivity status (connected vs. disconnected conditions)

**Groundwater pumping (±20%)**: Although concession data are available, actual extraction rates are often much lower than the permitted values.

**Groundwater outflow (±50%)** : Our Darcy-based estimate relies on assumed hydraulic conductivity values, which may vary by up to an order of magnitude.

## References
[\[1\]](#1---The-Limmat-Valley-Aquifer) Hug J., and Beilick, A. (1934): Die Grundwasserverhältnisse des Kantons Zürich. In: Beiträge zur Geologie der Schweiz - Geotechnische Serie - Hydrologie. German. Available online here: https://scnat.ch/de/uuid/i/0bd7aa3b-0bd7-5d54-9e2f-597a42dada50-Die_Grundwasserverh%C3%A4ltnisse_des_Kantons_Z%C3%BCrich (accessed 2025-05-01)      
[\[2\]](#1---The-Limmat-Valley-Aquifer) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.    
[\[3\]](#1---The-Limmat-Valley-Aquifer) GIS-browser of the canton of Zurich: https://www.gis.zh.ch/ (accessed 2025-05-01)    
[\[4\]](#22---Top-Topography) Federal Office of Topography swisstopo, swissALTI3D height model (2m resolution), [https://www.swisstopo.admin.ch/en/height-model-swissalti3d](https://www.swisstopo.admin.ch/en/height-model-swissalti3d) (accessed 2025-05-01)    
[\[5\]](#23---Lateral-Model-Boundaries) Freeze, R. A., and Cherry, J. A. (1979): Groundwater. Prentice-Hall, Englewood Cliffs, New Jersey, USA. ISBN: 978-0133653122. Open-book available here: [https://gw-project.org/books/groundwater/](https://gw-project.org/books/groundwater/)   
[\[6\]](#3---Climate-Forcing) MeteoSwiss: https://www.meteoswiss.admin.ch/services-and-publications/applications/measurement-values-and-measuring-networks.html#param=messnetz-klima&table=false&station=SMA&compare=y&chart=year (accessed 2025-05-01)  
[\[7\]](#3---Climate-Forcing) Opendata.swiss: Klimanormwerte, publisher: Federal Office of Meterology and Climatology. [https://opendata.swiss/en/dataset/klimanormwerte](https://opendata.swiss/en/dataset/klimanormwerte) (accessed 2025-05-01)    
[\[8\]](#3---Climate-Forcing) Lerner, David N. (2002): Identifying and quantifying urban recharge: a review. Hydrogeology Journal, Volume 10, Issue 1, pp 143–152. DOI: [https://doi.org/10.1007/s10040-001-0177-7](https://doi.org/10.1007/s10040-001-0177-1)    
[\[9\]](#3---Climate-Forcing) Qui, Guo Yu, Yan, Chunhua, Liu, Yuanbo (2023): Urban evapotranspiration and its effects on water budget and energy balance: Review and perspectives. Earth-Science Reviews, Volume 246. DOI: [https://doi.org/10.1016/j.earscirev.2023.104577](https://doi.org/10.1016/j.earscirev.2023.104577)       
[\[10\]](#4---River-aquifer-interaction) Locations of hydrological gauging stations maintained by the Federal Office for the Environment (FOEN): https://map.geo.admin.ch (accessed 2025-05-01)  
[\[11\]](#4---River-aquifer-interaction) Locations of hydrological gauging stations maintained by the cantonal office for waste, water, energy, and air (AWEL): https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/abfluss-wasserstand.html (accessed 2025-05-01)   
[\[12\]](#4---River-aquifer-interaction) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.   
[\[13\]](#5---Groundwater-pumping) City of Zurich, Drinking Water Supply: [https://www.stadt-zuerich.ch/de/umwelt-und-energie/wasser.html](https://www.stadt-zuerich.ch/de/umwelt-und-energie/wasser.html) (accessed 2025-08-21)    
[\[14\]](#5---Groundwater-pumping) Statistisches Jahrbuch der Stadt Zürich 2017 (Statistical yearbook of the city of Zurich 2017), Kapitel Wasser und Energie (Chapter water and energy), page 8: Wasserabgabe nach Wasserherkunft; Grundwasser (water deliveries by origin; groundwater).  [https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/publikationen-und-dienstleistungen/publikationen/jahrbuch.html](https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/publikationen-und-dienstleistungen/publikationen/jahrbuch.html) (accessed 2025-08-21)      
[\[13\]](#6---Monitoring-the-Limmat-Valley-Aquifer) Locations of groundwater monitoring wells maintained by Federal Office for the Environment (FOEN): https://www.zh.ch/de/umwelt-und-natur/wasser/grundwasser/monitoring.html (accessed 2025-05-01)  
[\[14\]](#6---Monitoring-the-Limmat-Valley-Aquifer) Cantonal office of the environment (AWEL): https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/grundwasserstaende.html (accessed 2025-05-01)